In [1]:
# Cell 0 — Project Imports

import torch
from torch import nn
from torch.nn import functional as F

In [2]:
# Cell 1 — Multi-Scale Deep-Supervision Logits

class TinyDeepSupervisionHead(nn.Module):
    """세 해상도의 segmentation logits 생성."""

    def __init__(self, in_channels: int, number_of_classes: int) -> None:
        super().__init__()
        self.full_head = nn.Conv3d(in_channels, number_of_classes, kernel_size=1)
        self.half_head = nn.Conv3d(in_channels, number_of_classes, kernel_size=1)
        self.quarter_head = nn.Conv3d(in_channels, number_of_classes, kernel_size=1)

    def forward(self, features: torch.Tensor) -> list[torch.Tensor]:  # [B,C,D,H,W]
        # 동일 feature를 세 spatial resolution으로 변환
        half = F.avg_pool3d(features, kernel_size=2, stride=2)  # [B,C,D/2,H/2,W/2]
        quarter = F.avg_pool3d(half, kernel_size=2, stride=2)  # [B,C,D/4,H/4,W/4]
        return [self.full_head(features), self.half_head(half), self.quarter_head(quarter)]

features = torch.randn(2, 8, 16, 32, 32)  # [B=2,C=8,D=16,H=32,W=32]
deep_supervision_head = TinyDeepSupervisionHead(in_channels=8, number_of_classes=4)
multi_scale_logits = deep_supervision_head(features)
for scale_index, logits in enumerate(multi_scale_logits):
    print(f"scale={scale_index} | logits Shape={list(logits.shape)}")

scale=0 | logits Shape=[2, 4, 16, 32, 32]
scale=1 | logits Shape=[2, 4, 8, 16, 16]
scale=2 | logits Shape=[2, 4, 4, 8, 8]


In [3]:
# Cell 2 — Deep-Supervision Target Resize

def resize_segmentation_target(
    target: torch.Tensor,       # [B,D,H,W], torch.long
    output_shape_zyx: tuple[int, int, int],
) -> torch.Tensor:             # [B,d,h,w], torch.long
    """Discrete class ID를 보존하는 nearest target resize."""
    if target.ndim != 4 or target.dtype != torch.long:
        raise TypeError("target은 [B,D,H,W] torch.long Tensor여야 합니다.")
    # Interpolate 입력을 위한 channel axis 추가 후 nearest resize
    resized = F.interpolate(
        target.unsqueeze(1).to(torch.float32),
        size=output_shape_zyx, mode="nearest",
    ).squeeze(1).to(torch.long)  # [B,d,h,w]
    return resized

full_target = torch.randint(0, 4, (2, 16, 32, 32), dtype=torch.long)  # [B,D,H,W]
multi_scale_targets = [
    resize_segmentation_target(full_target, tuple(logits.shape[-3:]))
    for logits in multi_scale_logits
]
for scale_index, target in enumerate(multi_scale_targets):
    print(f"scale={scale_index} | target Shape={list(target.shape)} | dtype={target.dtype}")

scale=0 | target Shape=[2, 16, 32, 32] | dtype=torch.int64
scale=1 | target Shape=[2, 8, 16, 16] | dtype=torch.int64
scale=2 | target Shape=[2, 4, 8, 8] | dtype=torch.int64


In [4]:
# Cell 3 — Weighted Deep-Supervision Loss

def compute_weighted_deep_supervision_loss(
    logits_list: list[torch.Tensor],     # 각 원소 [B,K,d,h,w]
    target_list: list[torch.Tensor],     # 각 원소 [B,d,h,w]
    scale_weights: torch.Tensor,         # [S], floating-point
) -> tuple[torch.Tensor, torch.Tensor]:  # Total [], per-scale [S]
    """각 해상도의 Cross-Entropy를 정규화된 weight로 결합."""
    if len(logits_list) != len(target_list) or scale_weights.shape != (len(logits_list),):
        raise ValueError("Scale 개수가 일치해야 합니다.")
    normalized_weights = scale_weights / scale_weights.sum()  # [S]
    losses = torch.stack([
        F.cross_entropy(logits, target)
        for logits, target in zip(logits_list, target_list)
    ])  # [S]
    return torch.sum(normalized_weights * losses), losses

scale_weights = torch.tensor([1.0, 0.5, 0.25], dtype=torch.float32)  # [S=3]
total_loss, per_scale_losses = compute_weighted_deep_supervision_loss(
    multi_scale_logits, multi_scale_targets, scale_weights
)
total_loss.backward()
print("Per-scale losses:", per_scale_losses.detach().tolist())
print("Normalized weights:", (scale_weights / scale_weights.sum()).tolist())
print("Total loss:", total_loss.item())
print("Full head gradient exists:", deep_supervision_head.full_head.weight.grad is not None)

Per-scale losses: [1.5335938930511475, 1.4339367151260376, 1.4033552408218384]
Normalized weights: [0.5714285969734192, 0.2857142984867096, 0.1428571492433548]
Total loss: 1.4865150451660156
Full head gradient exists: True


In [5]:
# Cell 4 — Sliding-Window 시작 좌표

def compute_sliding_window_starts(image_size: int, patch_size: int, step_fraction: float) -> tuple[int, ...]:
    """마지막 경계를 포함하는 1D sliding-window 시작점 계산."""
    if image_size <= 0 or patch_size <= 0 or patch_size > image_size:
        raise ValueError("0 < patch_size <= image_size 조건이 필요합니다.")
    if not 0.0 < step_fraction <= 1.0:
        raise ValueError("step_fraction은 (0,1] 범위여야 합니다.")
    if image_size == patch_size:
        return (0,)
    number_of_steps = int(torch.ceil(torch.tensor((image_size - patch_size) / (patch_size * step_fraction))).item()) + 1
    actual_step = (image_size - patch_size) / (number_of_steps - 1)
    return tuple(round(step_index * actual_step) for step_index in range(number_of_steps))

volume_shape_zyx = (20, 40, 40)
patch_shape_zyx = (12, 24, 24)
starts_zyx = tuple(
    compute_sliding_window_starts(image_size, patch_size, 0.5)
    for image_size, patch_size in zip(volume_shape_zyx, patch_shape_zyx)
)
print("Starts (z,y,x):", starts_zyx)
print("Number of windows:", len(starts_zyx[0]) * len(starts_zyx[1]) * len(starts_zyx[2]))

Starts (z,y,x): ((0, 4, 8), (0, 8, 16), (0, 8, 16))
Number of windows: 27


In [6]:
# Cell 5 — Gaussian-Weighted Logit Accumulation

def create_gaussian_importance_map(patch_shape_zyx: tuple[int, int, int]) -> torch.Tensor:  # [1,1,pD,pH,pW]
    """Patch 중앙의 신뢰도를 높이는 Gaussian weight 생성."""
    axes = [torch.linspace(-1.0, 1.0, steps=size) for size in patch_shape_zyx]
    grid_z, grid_y, grid_x = torch.meshgrid(*axes, indexing="ij")
    weights = torch.exp(-(grid_z.square() + grid_y.square() + grid_x.square()) / (2.0 * 0.5**2))
    return weights.clamp_min(1e-6).unsqueeze(0).unsqueeze(0)

# 각 window의 logits 합과 weight 합을 별도 buffer에 누적
number_of_classes = 3
logit_sum = torch.zeros(1, number_of_classes, *volume_shape_zyx)  # [B,K,D,H,W]
weight_sum = torch.zeros(1, 1, *volume_shape_zyx)  # [B,1,D,H,W]
importance_map = create_gaussian_importance_map(patch_shape_zyx)
for start_z in starts_zyx[0]:
    for start_y in starts_zyx[1]:
        for start_x in starts_zyx[2]:
            end_z, end_y, end_x = start_z + patch_shape_zyx[0], start_y + patch_shape_zyx[1], start_x + patch_shape_zyx[2]
            patch_logits = torch.ones(1, number_of_classes, *patch_shape_zyx)
            logit_sum[:, :, start_z:end_z, start_y:end_y, start_x:end_x] += patch_logits * importance_map
            weight_sum[:, :, start_z:end_z, start_y:end_y, start_x:end_x] += importance_map
# Overlap 영역을 weight 합으로 정규화
merged_logits = logit_sum / weight_sum.clamp_min(1e-8)  # [B,K,D,H,W]
print("Merged logits Shape:", list(merged_logits.shape))
print("Uncovered voxels:", int((weight_sum == 0).sum().item()))
print("All merged logits equal one:", bool(torch.allclose(merged_logits, torch.ones_like(merged_logits))))

Merged logits Shape: [1, 3, 20, 40, 40]
Uncovered voxels: 0
All merged logits equal one: True
